# Tabla base de discusiones tecnicas desde PDFs

Este cuaderno lee reportes expertos en PDF, extrae fragmentos candidatos y usa un LLM como skill/agente extractor para generar una tabla base con discusiones tecnicas verificables. No usa embeddings, FAISS, Chroma ni bases vectoriales.

La salida final de Excel contiene exactamente estas columnas: `Circuito`, `Fecha inicio`, `Fecha fin`, `Análisis`, `Evidencia`.

## 1. Configuracion inicial

In [1]:
import os
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

# Localizar raiz del proyecto y cargar configuracion activa.
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if load_dotenv:
    _env_file = ROOT / ".env"
    if _env_file.exists():
        load_dotenv(_env_file)
        print(f"Config cargada desde: {_env_file.name}")
    else:
        print("Config .env no encontrada; se usara entorno del sistema u Ollama si esta disponible.")

# Parametros principales del usuario
pdf_dir = ROOT / "reports" / "analysis-documents"
fecha_inicio_usuario = "2025-11-01"
fecha_fin_usuario = "2026-04-30"
output_excel = ROOT / "reports" / "analysis-documents" / "tabla_pdfs_intervalo_2025-11-01_2026-04-30.xlsx"

# Configuracion del LLM, siguiendo el patron de 02_local_uiti_vano_interpretability_v3.ipynb
CALL_LLM = os.getenv("CALL_LLM", "true").strip().lower() not in {"0", "false", "no", "off"}
_configured_provider = os.getenv("LLM_PROVIDER")
if _configured_provider:
    LLM_PROVIDER = _configured_provider
elif os.getenv("GOOGLE_API_KEY", "").strip():
    LLM_PROVIDER = "google"
elif os.getenv("OPENAI_API_KEY", "").strip():
    LLM_PROVIDER = "openai"
else:
    LLM_PROVIDER = "ollama"

if LLM_PROVIDER.lower() == "ollama":
    LLM_MODEL = os.getenv("LLM_MODEL") or os.getenv("OLLAMA_MODEL", "deepseek-r1:32b")
elif LLM_PROVIDER.lower() == "openai":
    LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4.1-mini")
else:
    LLM_MODEL = os.getenv("LLM_MODEL", "gemini-2.5-flash-lite")

LLM_MAX_OUTPUT_TOKENS = int(os.getenv("LLM_MAX_OUTPUT_TOKENS", "8192"))

# Segmentacion y preseleccion
CHUNK_CHARS = 6500
CHUNK_OVERLAP = 800
MAX_FRAGMENTOS = None  # usar un entero pequeno para pruebas rapidas

# Periodos generales conocidos por PDF. Si se deja vacio, el cuaderno intenta inferirlos del texto.
# Formato: {"reporte.pdf": ("YYYY-MM-DD", "YYYY-MM-DD")}
periodos_generales_por_pdf = {}

project_root = ROOT
artifact_root = project_root / "reports" / "interpretability" / "artifacts" / "pdf_discussion_extraction"
Path(output_excel).parent.mkdir(parents=True, exist_ok=True)
artifact_root.mkdir(parents=True, exist_ok=True)

print(f"PDF_DIR     : {pdf_dir}")
print(f"OUTPUT_XLSX : {output_excel}")
print(f"CALL_LLM    : {CALL_LLM}")
print(f"LLM_PROVIDER: {LLM_PROVIDER}")
print(f"LLM_MODEL   : {LLM_MODEL}")
print(f"LLM_MAX_OUT : {LLM_MAX_OUTPUT_TOKENS}")

Config cargada desde: .env
PDF_DIR     : /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/reports/analysis-documents
OUTPUT_XLSX : /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/reports/analysis-documents/tabla_pdfs_intervalo_2025-11-01_2026-04-30.xlsx
CALL_LLM    : True
LLM_PROVIDER: google
LLM_MODEL   : gemini-2.5-flash-lite
LLM_MAX_OUT : 8192


## 2. Carga de librerias

In [2]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from datetime import datetime
import json
import re
from typing import Any

import pandas as pd

from chec_local_interpreter.llm_client import call_llm

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

try:
    import pdfplumber
except ImportError:
    pdfplumber = None

COLUMNAS_FINALES = ["Circuito", "Fecha inicio", "Fecha fin", "Análisis", "Evidencia"]
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%dT%H%M%S")
run_dir = artifact_root / RUN_TIMESTAMP
run_dir.mkdir(parents=True, exist_ok=True)

## 3. Utilidades de fechas, texto y validacion

In [3]:
MESES = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "setiembre": "09", "octubre": "10",
    "noviembre": "11", "diciembre": "12",
}

TERMINOS_TECNICOS = [
    "falla", "apertura", "interrupción", "interrupcion", "recierre",
    "protección", "proteccion", "descarga atmosférica", "descarga atmosferica",
    "rayo", "vegetación", "vegetacion", "mantenimiento", "indisponibilidad",
    "MTTR", "MTBF", "cabecera", "ramal", "Gantt", "ventana", "causa",
    "recomendación", "recomendacion", "oscilografía", "oscilografia",
    "topología", "topologia", "maniobra", "evento", "afectación",
    "afectacion", "tramo", "circuito",
]

DATE_PATTERNS = [
    r"\b\d{4}-\d{2}-\d{2}\b",
    r"\b\d{1,2}/\d{1,2}/\d{4}\b",
    r"\b\d{1,2}-\d{1,2}-\d{4}\b",
    r"\b\d{1,2}\s+de\s+(?:" + "|".join(MESES) + r")\s+de\s+\d{4}\b",
    r"\b(?:" + "|".join(MESES) + r")\s+de\s+\d{4}\b",
]
DATE_REGEX = re.compile("|".join(DATE_PATTERNS), flags=re.IGNORECASE)
CIRCUIT_REGEX = re.compile(r"\b[A-Z]{2,5}\d{2}L\d{2}\b", flags=re.IGNORECASE)
PDF_CIRCUIT_REGEX = re.compile(r"(?<![A-Z0-9])[A-Z]{2,5}\d{2}L\d{2}(?![A-Z0-9])", flags=re.IGNORECASE)


def circuito_from_pdf_name(pdf_path: Path | str) -> str | None:
    stem = Path(pdf_path).stem.strip()
    if not stem or not PDF_CIRCUIT_REGEX.search(stem):
        return None
    return stem


def parse_fecha(value: str | None) -> pd.Timestamp | pd.NaT:
    if not value:
        return pd.NaT
    text = str(value).strip().lower()
    match = re.fullmatch(r"(\d{1,2})\s+de\s+([a-záéíóúñ]+)\s+de\s+(\d{4})", text)
    if match:
        day, month_name, year = match.groups()
        month = MESES.get(month_name)
        if month:
            return pd.to_datetime(f"{year}-{month}-{int(day):02d}", errors="coerce")
    match = re.fullmatch(r"([a-záéíóúñ]+)\s+de\s+(\d{4})", text)
    if match:
        month_name, year = match.groups()
        month = MESES.get(month_name)
        if month:
            return pd.to_datetime(f"{year}-{month}-01", errors="coerce")
    return pd.to_datetime(value, errors="coerce", dayfirst=True)


def iso_fecha(value: str | pd.Timestamp) -> str | None:
    parsed = parse_fecha(str(value)) if not isinstance(value, pd.Timestamp) else value
    if pd.isna(parsed):
        return None
    return parsed.strftime("%Y-%m-%d")


def overlaps(start: str, end: str, user_start: str, user_end: str) -> bool:
    start_ts = parse_fecha(start)
    end_ts = parse_fecha(end)
    user_start_ts = parse_fecha(user_start)
    user_end_ts = parse_fecha(user_end)
    if any(pd.isna(x) for x in [start_ts, end_ts, user_start_ts, user_end_ts]):
        return False
    return start_ts <= user_end_ts and end_ts >= user_start_ts


def extract_json_object(text: str) -> dict[str, Any] | None:
    if not text:
        return None
    cleaned = re.sub(r"```(?:json)?|```", "", text).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        try:
            return json.loads(cleaned[start:end + 1])
        except json.JSONDecodeError:
            return None


def validate_llm_row(payload: dict[str, Any], user_start: str, user_end: str) -> tuple[bool, list[str]]:
    errors = []
    if payload.get("include") is not True:
        return False, [payload.get("reason", "include=false")]
    for col in COLUMNAS_FINALES:
        if not str(payload.get(col, "")).strip():
            errors.append(f"{col} vacio")
    start = iso_fecha(payload.get("Fecha inicio"))
    end = iso_fecha(payload.get("Fecha fin"))
    if start is None:
        errors.append("Fecha inicio invalida")
    if end is None:
        errors.append("Fecha fin invalida")
    if start and end and parse_fecha(start) > parse_fecha(end):
        errors.append("Fecha inicio posterior a Fecha fin")
    if start and end and not overlaps(start, end, user_start, user_end):
        errors.append("La discusion no se traslapa con el rango del usuario")
    return len(errors) == 0, errors

## 4. Extraccion de texto y segmentacion

In [4]:
@dataclass(frozen=True)
class PDFPageText:
    nombre_pdf: str
    pagina: int
    texto: str


@dataclass(frozen=True)
class PDFFragment:
    nombre_pdf: str
    pagina_inicio: int
    pagina_fin: int
    periodo_general_informe: str
    fragmento: str


def extract_pdf_pages(pdf_path: Path) -> list[PDFPageText]:
    pages = []
    if fitz is not None:
        with fitz.open(pdf_path) as doc:
            for idx, page in enumerate(doc, start=1):
                text = page.get_text("text") or ""
                if not text.strip():
                    print(f"Advertencia: {pdf_path.name} pagina {idx} no tiene texto extraible.")
                pages.append(PDFPageText(pdf_path.name, idx, text))
        return pages
    if pdfplumber is not None:
        with pdfplumber.open(pdf_path) as doc:
            for idx, page in enumerate(doc.pages, start=1):
                text = page.extract_text() or ""
                if not text.strip():
                    print(f"Advertencia: {pdf_path.name} pagina {idx} no tiene texto extraible.")
                pages.append(PDFPageText(pdf_path.name, idx, text))
        return pages
    raise ImportError("Instala pymupdf o pdfplumber para extraer texto de PDFs.")


def detect_report_period(text: str) -> tuple[str, str] | None:
    fechas = [iso_fecha(match.group(0)) for match in DATE_REGEX.finditer(text)]
    fechas = [f for f in fechas if f]
    if len(fechas) < 2:
        return None
    sorted_dates = sorted(set(fechas))
    return sorted_dates[0], sorted_dates[-1]


def period_to_text(period: tuple[str, str] | None) -> str:
    if not period:
        return ""
    return f"{period[0]} a {period[1]}"


def chunk_pdf_pages(pages: list[PDFPageText], general_period: tuple[str, str] | None) -> list[PDFFragment]:
    fragments = []
    buffer = ""
    start_page = None
    last_page = None
    period_text = period_to_text(general_period)
    for page in pages:
        page_text = f"\n\n[Pagina {page.pagina}]\n{page.texto.strip()}"
        if start_page is None:
            start_page = page.pagina
        if len(buffer) + len(page_text) > CHUNK_CHARS and buffer.strip():
            fragments.append(PDFFragment(page.nombre_pdf, start_page, last_page or page.pagina, period_text, buffer.strip()))
            buffer = buffer[-CHUNK_OVERLAP:] if CHUNK_OVERLAP else ""
            start_page = last_page or page.pagina
        buffer += page_text
        last_page = page.pagina
    if buffer.strip() and start_page is not None:
        fragments.append(PDFFragment(pages[0].nombre_pdf, start_page, last_page or start_page, period_text, buffer.strip()))
    return fragments


def is_candidate_fragment(fragment: PDFFragment) -> bool:
    text = fragment.fragmento
    text_lower = text.lower()
    has_date = bool(DATE_REGEX.search(text)) or bool(fragment.periodo_general_informe)
    has_circuit = bool(CIRCUIT_REGEX.search(text))
    has_term = any(term.lower() in text_lower for term in TERMINOS_TECNICOS)
    return has_term and (has_date or has_circuit)


pdf_paths = sorted(Path(pdf_dir).glob("*.pdf"))
print(f"PDFs encontrados: {len(pdf_paths)}")

all_fragments = []
for pdf_path in pdf_paths:
    circuito_pdf = circuito_from_pdf_name(pdf_path)
    if circuito_pdf is None:
        print(f"{pdf_path.name}: omitido porque el nombre no contiene circuito.")
        continue
    pages = extract_pdf_pages(pdf_path)
    full_text = "\n".join(page.texto for page in pages)
    configured_period = periodos_generales_por_pdf.get(pdf_path.name)
    general_period = configured_period or detect_report_period(full_text)
    fragments = chunk_pdf_pages(pages, general_period)
    candidates = [fragment for fragment in fragments if is_candidate_fragment(fragment)]
    print(f"{pdf_path.name}: circuito={circuito_pdf}, {len(pages)} paginas, {len(fragments)} fragmentos, {len(candidates)} candidatos")
    all_fragments.extend(candidates)

if MAX_FRAGMENTOS is not None:
    all_fragments = all_fragments[:MAX_FRAGMENTOS]

print(f"Fragmentos candidatos totales: {len(all_fragments)}")

PDFs encontrados: 8
AGU23L15.pdf: circuito=AGU23L15, 19 paginas, 7 fragmentos, 7 candidatos
Advertencia: DON23L13.pdf pagina 30 no tiene texto extraible.
Advertencia: DON23L13.pdf pagina 31 no tiene texto extraible.
Advertencia: DON23L13.pdf pagina 32 no tiene texto extraible.
Advertencia: DON23L13.pdf pagina 33 no tiene texto extraible.
DON23L13.pdf: circuito=DON23L13, 45 paginas, 14 fragmentos, 14 candidatos


/var/folders/y9/9hfxsk810hv08cnh64bncjq80000gn/T/ipykernel_40825/56136799.py:53: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(value, errors="coerce", dayfirst=True)


INS23L13.pdf: circuito=INS23L13, 12 paginas, 5 fragmentos, 5 candidatos
MAZ23L13.pdf: circuito=MAZ23L13, 23 paginas, 11 fragmentos, 11 candidatos
Advertencia: MNA23L12.pdf pagina 8 no tiene texto extraible.
MNA23L12.pdf: circuito=MNA23L12, 8 paginas, 2 fragmentos, 2 candidatos
SNA23L12_V5.pdf: circuito=SNA23L12_V5, 30 paginas, 11 fragmentos, 11 candidatos
SNA23L15_V7.pdf: circuito=SNA23L15_V7, 21 paginas, 7 fragmentos, 7 candidatos
VMA23L13.pdf: circuito=VMA23L13, 24 paginas, 4 fragmentos, 4 candidatos
Fragmentos candidatos totales: 61


## 5. Skill/agente extractor

In [5]:
SKILL_PATH = project_root / "llm" / "skills_pdf_discussion_extraction" / "01_pdf_discussion_extractor.md"


class PDFDiscussionExtractionSkill:
    """Wrapper local que carga la skill versionada en llm/."""

    def __init__(
        self,
        skill_path: Path,
        provider: str,
        model: str,
        call_enabled: bool,
        max_output_tokens: int = 2048,
    ):
        self.skill_path = Path(skill_path)
        self.skill_template = self.skill_path.read_text(encoding="utf-8")
        self.provider = provider
        self.model = model
        self.call_enabled = call_enabled
        self.max_output_tokens = max_output_tokens

    def build_prompt(self, context: dict[str, Any]) -> str:
        prompt = self.skill_template
        for key, value in context.items():
            prompt = prompt.replace("{" + key + "}", str(value))
        return prompt.strip()

    def extract(self, context: dict[str, Any]) -> dict[str, Any]:
        prompt = self.build_prompt(context)
        result = call_llm(
            prompt,
            provider=self.provider,
            model=self.model,
            call_enabled=self.call_enabled,
            display_progress=False,
            display_content=False,
            max_output_tokens=self.max_output_tokens,
        )
        return {
            "prompt": prompt,
            "skill_path": str(self.skill_path),
            "context": context,
            "called": result.called,
            "message": result.message,
            "raw_output": result.output_text,
            "parsed": extract_json_object(result.output_text or ""),
        }

## 6. Ejecutar extraccion, validar y guardar artefactos

In [6]:
skill = PDFDiscussionExtractionSkill(
    skill_path=SKILL_PATH,
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    call_enabled=CALL_LLM,
    max_output_tokens=LLM_MAX_OUTPUT_TOKENS,
)

rows = []
invalid_records = []
jsonl_path = run_dir / "pdf_discussion_contexts_prompts_outputs.jsonl"

with jsonl_path.open("w", encoding="utf-8") as fh:
    for idx, fragment in enumerate(all_fragments, start=1):
        context = {
            "fecha_inicio_usuario": fecha_inicio_usuario,
            "fecha_fin_usuario": fecha_fin_usuario,
            "nombre_pdf": fragment.nombre_pdf,
            "circuito_pdf": circuito_from_pdf_name(fragment.nombre_pdf),
            "pagina_inicio": fragment.pagina_inicio,
            "pagina_fin": fragment.pagina_fin,
            "periodo_general_informe": fragment.periodo_general_informe,
            "fragmento": fragment.fragmento,
        }
        extraction = skill.extract(context)
        extraction["fragment_index"] = idx
        fh.write(json.dumps(extraction, ensure_ascii=False) + "\n")

        parsed = extraction.get("parsed")
        if not parsed:
            invalid_records.append({**extraction, "validation_errors": ["No se pudo parsear JSON valido"]})
            continue
        circuito_pdf = context["circuito_pdf"]
        if circuito_pdf is None:
            invalid_records.append({**extraction, "validation_errors": ["Nombre de PDF sin circuito"]})
            continue
        parsed["Circuito"] = circuito_pdf
        ok, errors = validate_llm_row(parsed, fecha_inicio_usuario, fecha_fin_usuario)
        if not ok:
            invalid_records.append({**extraction, "validation_errors": errors})
            continue

        row = {col: str(parsed[col]).strip() for col in COLUMNAS_FINALES}
        row["Fecha inicio"] = iso_fecha(row["Fecha inicio"])
        row["Fecha fin"] = iso_fecha(row["Fecha fin"])
        rows.append(row)

if invalid_records:
    invalid_path = run_dir / "invalid_llm_outputs.json"
    invalid_path.write_text(json.dumps(invalid_records, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Salidas invalidas o descartadas guardadas en: {invalid_path}")

print(f"Filas validas antes de deduplicar: {len(rows)}")

/var/folders/y9/9hfxsk810hv08cnh64bncjq80000gn/T/ipykernel_40825/56136799.py:53: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(value, errors="coerce", dayfirst=True)


Salidas invalidas o descartadas guardadas en: /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/reports/interpretability/artifacts/pdf_discussion_extraction/20260630T001311/invalid_llm_outputs.json
Filas validas antes de deduplicar: 14


## 7. Deduplicar y exportar Excel final

In [7]:
df_final = pd.DataFrame(rows, columns=COLUMNAS_FINALES)
if not df_final.empty:
    df_final = df_final.drop_duplicates(subset=COLUMNAS_FINALES).sort_values(
        by=["Circuito", "Fecha inicio", "Fecha fin", "Análisis", "Evidencia"]
    )

df_final = df_final.reindex(columns=COLUMNAS_FINALES)
df_final.to_excel(output_excel, index=False)

print(f"Excel generado: {output_excel}")
print(f"Filas finales: {len(df_final)}")
print(f"Artefactos de trazabilidad: {run_dir}")
df_final.head(20)

Excel generado: /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/reports/analysis-documents/tabla_pdfs_intervalo_2025-11-01_2026-04-30.xlsx
Filas finales: 14
Artefactos de trazabilidad: /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/reports/interpretability/artifacts/pdf_discussion_extraction/20260630T001311


,Circuito,Fecha inicio,Fecha fin,Análisis,Evidencia
2,AGU23L15,2016-01-01,2025-12-21,Se identifica una tendencia de degradación pos...,En la siguiente gráfica se muestra la cantidad...
1,AGU23L15,2025-09-23,2025-12-18,Se analizó el comportamiento de las señales de...,Se realizó un ejercicio del comportamiento de ...
0,AGU23L15,2025-12-21,2025-12-21,La mediana del MTTR de las maniobras No Progra...,La mediana del MTTR de las maniobras No Progra...
4,DON23L13,2021-01-01,2026-03-19,Se analiza la información del SCADA entre 2021...,se revisó la información del SCADA con 529.406...
5,DON23L13,2025-01-06,2026-04-30,Se observa una mejora en la severidad (horas a...,se observa una señal de mejora en severidad (h...
3,DON23L13,2025-05-02,2025-05-02,Mantenimiento programado en redes SDL asociado...,05-feb-2025: mantenimiento programado asociado...
6,DON23L13,2025-06-13,2025-06-13,Mejora en severidad (duración acumulada) poste...,Disminuye la duración acumulada posterior (mej...
7,DON23L13,2025-12-11,2026-10-02,Análisis de eventos y continuidad interna medi...,Figura E-5. Gantt ventana 90 días – 2025-11-12...
8,INS23L13,2025-10-21,2026-03-03,Mantenimientos en equipo C22969 (repotenciació...,Mantenimientos del 21 de octubre de 2025 y 3 d...
11,MAZ23L13,2025-04-15,2025-04-15,Intervención de mantenimiento asociada a reduc...,La intervención de mantenimiento del 15 de abr...
